# Project Integration: Fine-Tuned Resume Agent — Week 5

**Notebook:** `08_project_integration.ipynb`  
**Estimated time:** 30 minutes  

## Objectives
1. Wrap the fine-tuned model as a **gbrain-style skill** using `src.skill_wrapper`
2. Integrate HW4's RAG pipeline for retrieval-augmented generation
3. Run 5 end-to-end queries through the full agent pipeline
4. Author a `RESOLVER.md` documenting when to route to this skill

## Prerequisites
- NB07 complete: `outputs/merged_model/` or Ollama `hw5-finetuned` model registered
- (Optional) HW4 RAG pipeline at `../../Homework4-Submission/src/rag_pipeline.py`

---
## Introduction

In **Week 4** we built a RAG pipeline that retrieves relevant chunks from a PDF resume and passes them to an LLM. In **Week 5** we fine-tuned a model (`Qwen2.5-0.5B-Instruct`) to respond in a consistent, confident tone about resume content.

Now we combine them into a **gbrain-inspired agent skill**:

```
User query
    │
    ▼
RESOLVER (SkillResolver)
    │  routes based on keywords / intent
    ▼
hw5-resume-skill
    │  + retrieval context from RAG (if available)
    ▼
Fine-tuned Qwen2.5-0.5B (hw5-finetuned via Ollama)
    │
    ▼
Answer
```

This is the **gbrain architecture pattern**: `RESOLVER → skill dispatch → retrieval augmentation`. Each skill is a specialized model or tool; the resolver picks the right one based on the query. In production gbrain, skills are registered via `RESOLVER.md` manifests — we'll create one at the end of this notebook.

In [1]:
import sys
import importlib
import os
import json

sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
tracker = CostTracker()

print("Setup complete.")

Setup complete.


---
## Part 1: Building the Skill

A **skill** in gbrain is any callable that takes a `(prompt, context)` pair and returns a string answer. We wrap our fine-tuned model in a `FineTunedSkill` object that also carries metadata for the resolver: a name, a description, and keyword triggers.

In [2]:
import importlib
import src.skill_wrapper as _sw
importlib.reload(_sw)

from src.skill_wrapper import FineTunedSkill, SkillResolver, make_resume_skill
from src.llm_client import LLMClient

# Try to use the fine-tuned Ollama model; fall back to qwen3.5:27b
try:
    import ollama
    # Quick test to see if hw5-finetuned is registered
    _test = ollama.chat(
        model="hw5-finetuned",
        messages=[{"role": "user", "content": "ping"}],
    )
    model_fn = lambda prompt: ollama.chat(
        model="hw5-finetuned",
        messages=[{"role": "user", "content": prompt}],
    )["message"]["content"]
    print("Using fine-tuned hw5-finetuned model via Ollama")
except Exception as e:
    print(f"hw5-finetuned not available ({e}). Falling back to qwen3.5:27b via Ollama.")
    llm_client = LLMClient(path="B")  # Ollama fallback
    model_fn = lambda prompt: llm_client.generate(prompt)["content"]
    print("Fallback: using qwen3.5:27b")

# Build the resume skill
resume_skill = make_resume_skill(model_fn)

# Register skills with the resolver
resolver = SkillResolver([resume_skill])
resolver.show_skills()

Using fine-tuned hw5-finetuned model via Ollama
[skill_wrapper] Registered skill 'resume_qa' with 19 keywords
[skill_wrapper] SkillResolver initialized with 1 skill(s)
[skill_wrapper] Registered skills (1 total):
  [1] resume_qa
       Description: Answers questions about professional experience, skills, education, and career history from a resume.
       Keywords: resume, experience, skills, education, career, job, work, background, qualification, degree, university, company, position, role, project, achievement, certification, profile, candidate


---
## Part 2: Adding RAG Context (gbrain Pattern)

In gbrain, every skill can receive **retrieval context** from the knowledge graph. We replicate this pattern using our HW4 RAG modules: when a user asks a resume question, we first retrieve the top-3 relevant chunks from the PDF, then pass them as context to the fine-tuned model.

This is more powerful than either approach alone:
- **RAG alone** gives factual grounding but generic phrasing
- **Fine-tuning alone** gives the right tone but may hallucinate facts
- **RAG + fine-tuning** gives accurate facts in the right tone

In [8]:
# Try to load HW4 RAG pipeline
sys.path.insert(0, '..')

try:
    from src.rag_pipeline import RAGPipeline
    rag = RAGPipeline.from_pdf("../test_data/sample_resume.pdf")
    has_rag = True
    print("HW4 RAG pipeline loaded successfully")
except Exception as e:
    has_rag = False
    print(f"RAG not available: {e}")
    print("Continuing in skill-only mode (no retrieval context).")


def query_with_rag(question: str) -> str:
    """Route a question through the resolver, optionally enriching with RAG context."""
    if has_rag:
        ctx = rag.retrieve(question, top_k=3)
        retrieval_ctx = "\n".join([c["text"] for c in ctx])
    else:
        retrieval_ctx = ""
    return resolver.dispatch(question, retrieval_ctx=retrieval_ctx)


print(f"\nquery_with_rag ready (RAG enabled: {has_rag})")

  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pypdf)
  Loading all-MiniLM-L6-v2 (sentence-transformers, cpu)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/workspace/Homework5-Submission/notebooks/../src/embeddings.py:83: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._client.get_sentence_embedding_dimension()


  ✓ Loaded in 4.7s, dim=384
  ✓ recursive_chunk: 5 chunks (size~500, overlap=50)
  ✓ recursive_chunk: 5 chunks (size~500, overlap=50)

  Embedding 5 chunks...
  ✓ FAISS: 5 vectors added (total: 5)
HW4 RAG pipeline loaded successfully

query_with_rag ready (RAG enabled: True)


---
## Part 3: End-to-End Agent Run

Let's run 5 representative resume queries through the full pipeline and inspect the answers.

In [9]:
test_queries = [
    "What programming languages does Scott know?",
    "What was Scott's most recent job?",
    "Does Scott have experience with machine learning?",
    "What is Scott's educational background?",
    "What kind of roles is Scott looking for?",
]

print(f"Running {len(test_queries)} queries through the full agent pipeline...")
print("=" * 70)

for i, q in enumerate(test_queries, 1):
    print(f"\n[Query {i}/{len(test_queries)}]")
    print(f"Q: {q}")
    answer = query_with_rag(q)
    preview = answer[:200]
    print(f"A: {preview}{'...' if len(answer) > 200 else ''}")
    print("-" * 70)

Running 5 queries through the full agent pipeline...

[Query 1/5]
Q: What programming languages does Scott know?
[skill_wrapper] Resolving query: What programming languages does Scott know?...
[skill_wrapper] No keyword match, falling back to 'resume_qa'
[skill_wrapper] Skill 'resume_qa' invoked for query: What programming languages does Scott know?...
[skill_wrapper] Skill 'resume_qa' returned 34 chars
A: Scott knows Python, Java, and SQL.
----------------------------------------------------------------------

[Query 2/5]
Q: What was Scott's most recent job?
[skill_wrapper] Resolving query: What was Scott's most recent job?...
[skill_wrapper] Resolved to skill 'resume_qa' (score=1.0)
[skill_wrapper] Skill 'resume_qa' invoked for query: What was Scott's most recent job?...
[skill_wrapper] Skill 'resume_qa' returned 56 chars
A: Scott's most recent job is Python Developer at DoorDash.
----------------------------------------------------------------------

[Query 3/5]
Q: Does Scott have e

---
## Part 4: RESOLVER.md — gbrain Documentation Pattern

In gbrain, each skill is documented in a `RESOLVER.md` manifest. This file tells the orchestrator:
- **When** to route to this skill (what kinds of questions it handles)
- **What keywords** signal this skill is relevant
- **Example queries** for few-shot routing
- **Fallback** behavior when no skill matches

Let's generate one for our resume skill:

In [19]:
resolver_md = """# RESOLVER.md — Skill Routing Guide

## hw5-resume-skill
**When to use:** Queries about resume content, work history, skills, education, career goals  
**Keywords:** resume, experience, job, background, skills, education, worked, built, achieved  
**Example queries:**
- "What languages does Scott know?"
- "Tell me about Scott's experience at [company]"
- "What is Scott's highest degree?"
- "What projects has Scott built?"
- "What kind of roles is Scott targeting?"

**Model:** hw5-finetuned (Qwen2.5-0.5B-Instruct, fine-tuned on synthetic resume Q&A)  
**RAG:** Yes — retrieves top-3 chunks from sample_resume.pdf before generating  

## Fallback
If no skill matches, route to the base LLM (qwen3.5:27b via Ollama).  
Trigger condition: query does not contain resume-related keywords AND no skill confidence > 0.5.
"""

os.makedirs("../outputs", exist_ok=True)
with open("../outputs/RESOLVER.md", "w") as f:
    f.write(resolver_md)

print(resolver_md)

# RESOLVER.md — Skill Routing Guide

## hw5-resume-skill
**When to use:** Queries about resume content, work history, skills, education, career goals  
**Keywords:** resume, experience, job, background, skills, education, worked, built, achieved  
**Example queries:**
- "What languages does Scott know?"
- "Tell me about Scott's experience at [company]"
- "What is Scott's highest degree?"
- "What projects has Scott built?"
- "What kind of roles is Scott targeting?"

**Model:** hw5-finetuned (Qwen2.5-0.5B-Instruct, fine-tuned on synthetic resume Q&A)  
**RAG:** Yes — retrieves top-3 chunks from sample_resume.pdf before generating  

## Fallback
If no skill matches, route to the base LLM (qwen3.5:27b via Ollama).  
Trigger condition: query does not contain resume-related keywords AND no skill confidence > 0.5.



---
## TODO 1: Add a Second Skill

The `SkillResolver` currently has only one skill (`hw5-resume-skill`). Add a **"general Q&A" skill** that:
- Uses the Ollama base model (`qwen3.5:27b`) for any query the resume skill doesn't match
- Has different keyword triggers (e.g., "explain", "what is", "how does")
- Is registered with the resolver alongside the resume skill

Then test that the resolver routes correctly:
- `"What is Scott's background?"` → resume skill
- `"What is gradient descent?"` → general Q&A skill

**Starter code:**

In [20]:
# TODO 1: Add a general Q&A skill and test routing

# Example:
# general_client = LLMClient(path="B")  # Ollama qwen3.5:27b
# general_fn = lambda prompt: general_client.generate(prompt)["content"]
#
# general_skill = FineTunedSkill(
#     name="general-qa",
#     description="Answers general knowledge questions not related to the resume",
#     model_fn=general_fn,
#     keywords=["explain", "what is", "how does", "why", "define", "describe"],
# )
#
# multi_resolver = SkillResolver([resume_skill, general_skill])
# multi_resolver.show_skills()
#
# # Test routing
# print(multi_resolver.dispatch("What is Scott's educational background?"))
# print(multi_resolver.dispatch("What is gradient descent?"))

general_client = LLMClient(path="B")

available_models = general_client.get_available_models()
if "qwen3.5:27b" in available_models:
  general_model = "qwen3.5:27b"
elif "qwen3.5:9b" in available_models:
  general_model = "qwen3.5:9b"
else:
  raise ValueError(f"No qwen3.5 base model found. Available models: {available_models}")

print(f"Using general Q&A model: {general_model}")

general_fn = lambda prompt: general_client.generate(
  prompt,
  model=general_model,
  system="You are a concise general Q&A assistant. Answer directly.",
  temperature=0.2,
  max_tokens=300,
)["content"]

general_skill = FineTunedSkill(
  name="general-qa",
  description="Answers general knowledge questions not related to the resume",
  model_fn=general_fn,
  match_keywords=["explain", "what is", "how does", "why", "define", "describe"],
)

multi_resolver = SkillResolver([resume_skill, general_skill])
multi_resolver.show_skills()

resume_test = "What is Scott's background?"
general_test = "What is gradient descent?"

print(f"\nRoute test 1: {resume_test!r}")
print("Resolved skill:", multi_resolver.resolve(resume_test).name)
print(multi_resolver.dispatch(resume_test))

print("\n" + "-" * 70)

print(f"\nRoute test 2: {general_test!r}")
print("Resolved skill:", multi_resolver.resolve(general_test).name)
print(multi_resolver.dispatch(general_test))

print("\n" + "-" * 70)
todo1_reflection = "I added a second general-qa skill with triggers such as 'what is', 'explain', and 'how does', then registered it alongside the resume skill. The resolver routed 'What is Scott's background?' to the resume skill because it matched  the resume/background keywords. It routed 'What is gradient descent?' to  general-qa because it matched the general 'what is' trigger and did not match  resume-specific keywords. This confirms that resume questions still go to the fine-tuned resume skill while unrelated knowledge questions use the base Ollama model."
print(todo1_reflection)

✓ Ollama client initialized
  Available models: ['hw5-finetuned:latest', 'qwen3.5:9b']
  Default model: hw5-finetuned:latest
Using general Q&A model: qwen3.5:9b
[skill_wrapper] Registered skill 'general-qa' with 6 keywords
[skill_wrapper] SkillResolver initialized with 2 skill(s)
[skill_wrapper] Registered skills (2 total):
  [1] resume_qa
       Description: Answers questions about professional experience, skills, education, and career history from a resume.
       Keywords: resume, experience, skills, education, career, job, work, background, qualification, degree, university, company, position, role, project, achievement, certification, profile, candidate
  [2] general-qa
       Description: Answers general knowledge questions not related to the resume
       Keywords: explain, what is, how does, why, define, describe

Route test 1: "What is Scott's background?"
[skill_wrapper] Resolving query: What is Scott's background?...
[skill_wrapper] Resolved to skill 'resume_qa' (score=1.0)


---
## TODO 2: Project Update

Write your weekly project update in the cell below, then run the summary cell to save it.

**Template:**
```markdown
# Week 5 Project Update — [Your Name]
## What I built this week
## How Week 5 connects to Week 4 (RAG + fine-tuning)
## What surprised me most about fine-tuning
## What I would improve with more compute/time
```

In [21]:
# TODO 2: Fill in your project update
project_update = """
# Week 5 Project Update — Kai Yang

## What I built this week
This week I built a fine-tuned resume assistant. I trained a LoRA adapter on Qwen2.5-0.5B-Instruct using synthetic resume Q&A data, tried a small DPO preference-tuning step, merged the model, converted it to
GGUF, and served it locally with Ollama as hw5-finetuned. I also wrapped it as a gbrain-style skill and added a general Q&A skill so the resolver can route resume questions and normal knowledge questions
separately.

## How Week 5 connects to Week 4 (RAG + fine-tuning)
Week 4 handled retrieval: the RAG pipeline pulls relevant chunks from the resume PDF. Week 5 adds the fine-tuned model on top, so the assistant can use that retrieved context and answer in a more polished,
resume-focused style. RAG helps keep the facts grounded, while fine-tuning helps with tone and consistency.

## What surprised me most about fine-tuning
I was surprised that a lower training loss did not automatically mean the answers were fully reliable. The fine-tuned model sounded more like a resume assistant, but it could still hallucinate when the context
was weak. My eval improved only a little, from 1.80 for the baseline to 2.00 for the fine-tuned model, which showed me that good data and retrieval matter a lot.

## What I would improve with more compute/time
I would use more verified training examples, add a real validation set, and make the DPO dataset much larger. I would also make RAG required for resume questions and improve the resolver so it can be more
confident about when to use the resume skill versus the general Q&A skill.
"""

print(project_update)


# Week 5 Project Update — Kai Yang

## What I built this week
This week I built a fine-tuned resume assistant. I trained a LoRA adapter on Qwen2.5-0.5B-Instruct using synthetic resume Q&A data, tried a small DPO preference-tuning step, merged the model, converted it to
GGUF, and served it locally with Ollama as hw5-finetuned. I also wrapped it as a gbrain-style skill and added a general Q&A skill so the resolver can route resume questions and normal knowledge questions
separately.

## How Week 5 connects to Week 4 (RAG + fine-tuning)
Week 4 handled retrieval: the RAG pipeline pulls relevant chunks from the resume PDF. Week 5 adds the fine-tuned model on top, so the assistant can use that retrieved context and answer in a more polished,
resume-focused style. RAG helps keep the facts grounded, while fine-tuning helps with tone and consistency.

## What surprised me most about fine-tuning
I was surprised that a lower training loss did not automatically mean the answers were fully reliabl

---
## Summary

In [23]:
from datetime import datetime

# Save project update
os.makedirs("../outputs", exist_ok=True)
update_content = project_update.strip() if 'project_update' in dir() else "[TODO: fill in]"
with open("../outputs/my_project_update.md", "w") as f:
    f.write(update_content)
print("Project update saved to ../outputs/my_project_update.md")

# Summarize outputs
outputs = [
    "../outputs/RESOLVER.md",
    "../outputs/my_project_update.md",
]
print("\n=== NB08 Outputs ===")
for path in outputs:
    exists = os.path.exists(path)
    print(f"  {'[OK]' if exists else '[MISSING]'} {path}")

# Append to reflection log
def append_to_reflection(nb_id: str, nb_title: str, reflection: str, path: str = "../outputs/reflection_log.json"):
    log = []
    if os.path.exists(path):
        with open(path) as f:
            log = json.load(f)
    log.append({
        "notebook": nb_id,
        "title": nb_title,
        "reflection": reflection,
        "timestamp": datetime.now().isoformat(),
    })
    with open(path, "w") as f:
        json.dump(log, f, indent=2)
    print(f"Reflection appended to {path}")

append_to_reflection(
    "08",
    "Project Integration",
    todo1_reflection if 'todo1_reflection' in dir() else "[TODO 1 not completed]",
)

tracker.report()

Project update saved to outputs/my_project_update.md

=== NB08 Outputs ===
  [OK] ../outputs/RESOLVER.md
  [OK] ../outputs/my_project_update.md
Reflection appended to ../outputs/reflection_log.json
API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

